<a href="https://colab.research.google.com/github/aviyadav/end-to-end-data-engineering-project-FMCG/blob/main/DE_Project_FMCG.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import os
import pandas as pd
import numpy as np
import random
from datetime import datetime, timedelta
from google.colab import drive

# 1. Google Drive's connection
drive.mount('/content/drive')

# 2. Setting Up the Folder Structure (Medallion Architecture)
base_path = '/content/drive/MyDrive/DE_Project_FMCG'
folders = {
    'landing': os.path.join(base_path, 'Landing'),  # raw CSV
    'bronze': os.path.join(base_path, 'Bronze'),    # Arcive (parquet)
    'silver': os.path.join(base_path, 'Silver'),    # Cleaned Data
    'gold': os.path.join(base_path, 'Gold'),    # Reporting Data
}

for layer, path in folders.items():
    os.makedirs(path, exist_ok=True)

# 3. Creating the Raw (Dirty) Dataset
# Contains incorrect city names, spaces, and mixed date formats.
customer_ids = range(101, 151)
customers = []
cities_messy = ['Istanbul', 'Istnbul', 'Ankara', 'Ankr', 'Izmir', None, '  Bursa  ']

for cid in customer_ids:
  name = "Customer " + str(cid)
  if random.random() < 0.3: name = " " + name + " "   # Trim error
  customers.append([cid, name, random.choice(cities_messy), 'TR'])

df_customers = pd.DataFrame(customers, columns=['customer_id', 'customer_name', 'city', 'market'])

# Product and Order data are created and saved with similar logic (incorrectly)...
# (The full code is available in the steps above)

df_customers.to_csv(os.path.join(folders['landing'], 'customers.csv'), index=False)
# ... Other files are also saved to Landing.

print("✅ Infrastructure is ready and dirty data has been uploaded to the Landing folder.")

Mounted at /content/drive
✅ Infrastructure is ready and dirty data has been uploaded to the Landing folder.


In [3]:
product_ids = range(1, 21)
products = []
product_categories = ['Electronics', 'Home Goods', 'Apparel', 'Books', 'Food']

for pid in product_ids:
    name = f"Product {pid}"
    category = random.choice(product_categories)
    price = round(random.uniform(5.0, 500.0), 2)
    products.append([pid, name, category, price])

df_products = pd.DataFrame(products, columns=['product_id', 'product_name', 'category', 'price'])
df_products.to_csv(os.path.join(folders['landing'], 'products.csv'), index=False)
print("✅ Products data generated and uploaded to Landing folder.")

✅ Products data generated and uploaded to Landing folder.


In [4]:
order_ids = range(1001, 1501)
orders = []

start_date = datetime(2023, 1, 1)
end_date = datetime(2023, 12, 31)

for oid in order_ids:
    customer_id = random.choice(customer_ids) # Using customer_ids from previous cell
    product_id = random.choice(product_ids) # Using product_ids from previous cell
    quantity = random.randint(1, 5)
    order_date = start_date + timedelta(days=random.randint(0, (end_date - start_date).days))
    orders.append([oid, customer_id, product_id, quantity, order_date.strftime('%Y-%m-%d')]) # Format date as string

df_orders = pd.DataFrame(orders, columns=['order_id', 'customer_id', 'product_id', 'quantity', 'order_date'])
df_orders.to_csv(os.path.join(folders['landing'], 'orders.csv'), index=False)
print("✅ Orders data generated and uploaded to Landing folder.")

✅ Orders data generated and uploaded to Landing folder.


In [5]:
def ingest_to_bronze(file_name, source_folder, target_folder):
    source_path = os.path.join(source_folder, f"{file_name}.csv")
    target_path = os.path.join(target_folder, f"{file_name}.parquet")

    try:
        # Read CSV
        df = pd.read_csv(source_path)

        # Add Audit Columns
        df['_ingestion_timestamp'] = datetime.now()
        df['_source_file'] = f"{file_name}.csv"

        # Save as Parquet
        df.to_parquet(target_path, index=False)
        print(f"✅ {file_name} -> Moved to Bronze layer.")

    except Exception as e:
        print(f"❌ Error: {e}")

# Process all files
files = ['customers', 'products', 'orders']
for f in files:
    ingest_to_bronze(f, folders['landing'], folders['bronze'])

✅ customers -> Moved to Bronze layer.
✅ products -> Moved to Bronze layer.
✅ orders -> Moved to Bronze layer.


In [6]:
import hashlib
import re

# --- Customer Cleaning ---
df_cust = pd.read_parquet(os.path.join(folders['bronze'], 'customers.parquet'))
df_cust['customer_name'] = df_cust['customer_name'].str.strip() # Trim
# City correction map
city_map = {'Istnbul': 'Istanbul', 'Ankr': 'Ankara', '  Bursa  ': 'Bursa'}
df_cust['city'] = df_cust['city'].replace(city_map).fillna('Unknown').str.strip()
df_cust.to_parquet(os.path.join(folders['silver'], 'dim_customers.parquet'), index=False)

# --- Product Cleaning (Regex & Hashing) ---
df_prod = pd.read_parquet(os.path.join(folders['bronze'], 'products.parquet'))

# Variant Extraction with Regex (Ex: "Protein Bar 50g" -> "50g")
def extract_variant(text):
    match = re.search(r'(\d+(?:g|kg|ml|tabs))', text, re.IGNORECASE)
    return match.group(0) if match else 'Regular'

df_prod['variant'] = df_prod['product_name'].apply(extract_variant)

# New ID Generation with SHA-256
df_prod['product_code'] = df_prod['product_name'].apply(lambda x: hashlib.sha256(x.encode()).hexdigest()[:16])
df_prod.to_parquet(os.path.join(folders['silver'], 'dim_products.parquet'), index=False)

# --- Order Cleaning (Date Standardization) ---
df_ord = pd.read_parquet(os.path.join(folders['bronze'], 'orders.parquet'))
# Convert mixed formats (2024/01/01 vs 01-01-2024) to a single format with Pandas
df_ord['order_date'] = pd.to_datetime(df_ord['order_date'], format='mixed', errors='coerce')
df_ord = df_ord.dropna(subset=['order_date']) # Delete incorrect dates
df_ord.to_parquet(os.path.join(folders['silver'], 'fact_orders.parquet'), index=False)

print("🚀 Silver Layer (Clean Data) Ready!")

🚀 Silver Layer (Clean Data) Ready!


In [7]:
# Read Silver Data
df_orders = pd.read_parquet(os.path.join(folders['silver'], 'fact_orders.parquet'))
df_cust = pd.read_parquet(os.path.join(folders['silver'], 'dim_customers.parquet'))
df_prod = pd.read_parquet(os.path.join(folders['silver'], 'dim_products.parquet'))

# Merge Tables (Star Schema -> Wide Table)
df_gold = pd.merge(df_orders, df_cust, on='customer_id', how='left')
df_gold = pd.merge(df_gold, df_prod, on='product_id', how='left')

# Data Enrichment (Calculated Column)
df_gold['revenue'] = df_gold['quantity'] * df_gold['price']

# Simplification for report and Null management
df_gold['customer_name'] = df_gold['customer_name'].fillna('Unknown Customer')
final_cols = ['order_date', 'city', 'category', 'product_name', 'variant', 'revenue', 'quantity']
df_gold_final = df_gold[final_cols]

# Save Results
# 1. Parquet: For modern analyses
df_gold_final.to_parquet(os.path.join(folders['gold'], 'report_master_table.parquet'), index=False)
# 2. CSV: For classic tools (Excel/Tableau)
df_gold_final.to_csv(os.path.join(folders['gold'], 'report_master_table.csv'), index=False)

print("✅ Reporting Table Created.")

✅ Reporting Table Created.


In [8]:
!pip install duckdb --quiet

In [9]:
import duckdb

parquet_file = os.path.join(folders['gold'], 'report_master_table.parquet')

# SQL 1: Total Revenue by City
query_city = f"""
    SELECT
        city,
        ROUND(SUM(revenue), 2) as total_revenue
    FROM '{parquet_file}'
    GROUP BY city
    ORDER BY total_revenue DESC
"""
df_analysis = duckdb.query(query_city).to_df()
print("--- DuckDB Analysis Result ---")
display(df_analysis)

# SQL 2: Category-Based Sales
query_category = f"""
    SELECT
        category,
        SUM(quantity) as total_quantity,
        ROUND(AVG(revenue), 2) as avg_basket_amount
    FROM '{parquet_file}'
    GROUP BY category
    ORDER BY total_quantity DESC
"""
display(duckdb.query(query_category).to_df())

--- DuckDB Analysis Result ---


,city,total_revenue
0,Ankara,128715.14
1,Istanbul,79831.32
2,Bursa,74341.89
3,Unknown,67508.25
4,Izmir,37318.50


,category,total_quantity,avg_basket_amount
0,Books,483.0,877.86
1,Apparel,357.0,738.34
2,Food,286.0,626.64
3,Electronics,234.0,635.66
4,Home Goods,170.0,1029.44
